<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/Leonardo-branch/coleta_DATASUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Retire o comentário desse comando e execute ele para limpar o diretório caso necessário
#!rm -rf /content/crise-saude-ufam


In [ ]:
import json, time, datetime, pathlib, unicodedata, shutil, glob, os
from getpass import getpass
from google.colab import userdata
from pathlib import Path
import pyarrow.parquet as pq
import httpx

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

/usr/local/lib/python3.13/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.6.0)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


pandas  2.2.3
requests 2.32.4


In [ ]:
USUARIO = "victorbaraunaAcad"
REPO    = "crise-saude-ufam"

'''
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/{USUARIO}/{REPO}.git
%cd {REPO}
!git config user.name "gabrielsanttos"
!git config user.email "gabrielconceicao827@gmail.com"
'''
RAIZ     = pathlib.Path.cwd()
BRUTOS   = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BRUTOS.mkdir(exist_ok=True); TRATADOS.mkdir(exist_ok=True)
print("Trabalhando em:", RAIZ)

Cloning into 'crise-saude-ufam'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 61 (delta 33), reused 22 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 64.77 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/crise-saude-ufam
Trabalhando em: /content/crise-saude-ufam


In [ ]:

%pip install pysus
import pysus

%pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

%pip install duckdb -q
import duckdb
con = duckdb.connect()

print(f"pysus {pysus.get_version()}")

pysus 2.11.2


In [ ]:
ARQ_PROV = RAIZ / "proveniencia.csv"
RAIZ_BRUTOS = pathlib.Path("/content/crise-saude-ufam/dados_brutos")
RAIZ_BRUTOS.mkdir(parents=True, exist_ok=True)

def registrar(fonte, url, metodo, parametros, n_linhas, arquivo_bruto):
    linha = pd.DataFrame([{
        "fonte": fonte, "url": url, "metodo": metodo,
        "parametros": json.dumps(parametros, ensure_ascii=False),
        "n_linhas": n_linhas,
        "arquivo_bruto": str(pathlib.Path(arquivo_bruto).relative_to(RAIZ)),
        "coletado_em": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    }])
    cabecalho = not ARQ_PROV.exists() or ARQ_PROV.stat().st_size == 0
    linha.to_csv(ARQ_PROV, mode="a", header=cabecalho, index=False)
    print(f"  ✓ proveniência: {fonte} ({n_linhas} linhas)")

def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.upper().strip()

In [ ]:
#Definindo as funções que faz o download dos dados
from pysus import list_files

CODIGO_UF = {
    "AM": "13", "RS": "43", "MS": "50","MT": "51", "GO": "52", "DF": "53",
}

def baixar_do_estado(nome_base, funcao, uf, ano, coluna_municipio="CODMUNRES"):
    """Baixa a base e garante que só sobre o estado pedido."""
    catalogo = list_files(dataset=nome_base, state=uf, year=ano)
    do_estado = {
        pathlib.Path(str(n)).name
        for n in catalogo.loc[catalogo["state"] == uf, "name"]
    }
    #salva a url de onde os dados foram baixados
    urls = [f"ftp://ftp.datasus.gov.br/{n}"
            for n in catalogo.loc[catalogo["state"] == uf, "name"]]


    caminhos = funcao(state=uf, year=ano)
    escolhidos = [p for p in caminhos if pathlib.Path(str(p)).name in do_estado]

    if escolhidos:
        df = pd.concat([pd.read_parquet(p) for p in escolhidos], ignore_index=True)
        arquivos_para_copiar = escolhidos
    else:
        df = pd.concat([pd.read_parquet(p) for p in caminhos], ignore_index=True)
        antes = len(df)
        df = df[df[coluna_municipio].astype(str).str[:2] == CODIGO_UF[uf]]
        print(f"  (este ano só tem arquivo nacional: filtrei {antes:,} linhas "
              f"do país para {len(df):,} de {uf})")
        arquivos_para_copiar = escolhidos


    df = df.reset_index(drop=True)

    # copia os arquivos originais do PySUS (tal como vieram) para dados_brutos/
    caminhos_copiados = []
    for origem in arquivos_para_copiar:
        destino = RAIZ_BRUTOS / pathlib.Path(str(origem)).name
        shutil.copy(origem, destino)
        caminhos_copiados.append(str(destino))


    registrar(
        fonte=nome_base,
        url="; ".join(urls),
        metodo=funcao.__name__,
        parametros={"state": uf, "year": ano},
        n_linhas=len(df),
        arquivo_bruto="; ".join(caminhos_copiados),
    )

    return df


#Função para baixar dados de várias UFs de uma vez e concatenar, utilizada para
def baixar_regiao(nome_base, funcao, ufs, ano, coluna_municipio="CODMUNRES"):
    """Baixa e junta os dados de várias UFs em uma única tabela."""
    dfs = []
    for uf in ufs:
        print(f"Baixando {nome_base} - {uf} - {ano}...")
        df_uf = baixar_do_estado(nome_base, funcao, uf, ano, coluna_municipio)
        df_uf["UF"] = uf  # garante rastreabilidade, mesmo que o código já traga o estado
        dfs.append(df_uf)
    return pd.concat(dfs, ignore_index=True)



In [ ]:
#Primeira coleta: SIM(Sistema de Informações sobre Mortalidade)
from pysus import sim
#Primeira coleta: UF = Amazonas, Período: [2023,2024]
UF = "AM"
ANO = [2023,2024]


obitos_am = baixar_do_estado("sim", sim, UF,ANO)

#Segunda coleta: UF= Rio Grande do Sul, Período = 2024
UF = "RS"
ANO = 2024

obitos_rs = baixar_do_estado("sim",sim, UF,ANO)

#Terceira coleta: UFs= Mato Grosso do Sul, Mato Grosso, Goiás, Distrito Federal, Período = 2024
UF = ["MS","MT", "GO", "DF"]
ANO = 2024

obitos_centro_oeste = baixar_regiao("sim", sim, UF, ANO)





#Informações sobre os arquivos baixados
print(f"{len(obitos_am):,} óbitos registrados em AM, 2023-2024")
print(f"{len(obitos_am.columns)} colunas disponíveis")

print(f"{len(obitos_rs):,} óbitos registrados em RS, 2024")
print(f"{len(obitos_rs.columns)} colunas disponíveis")



print(f"{len(obitos_centro_oeste):,} óbitos registrados em MS, MT, GO, DF, 2024")
print(f"{len(obitos_centro_oeste.columns)} colunas disponíveis")



/tmp/ipykernel_38962/123103405.py:10: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset=nome_base, state=uf, year=ano)
/tmp/ipykernel_38962/123103405.py:20: PySUSWarning: pysus.sim() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sim(...), pysus.dadosgov.sim(...), or pysus.saude.sim(...). Behavior is unchanged.
  caminhos = funcao(state=uf, year=ano)
DOAM2024.parquet: 0.00B [00:00, ?B/s]

DOAM2023.parquet: 0.00B [00:00, ?B/s]


DOAM2023.parquet: 0.00B [00:00, ?B/s]
DOAM2024.parquet: 0.00B [00:00, ?B/s]

DO24OPEN.parquet: 0.00B [00:00, ?B/s]
DO23OPEN.parquet: 0.00B [00:00, ?B/s]


  ✓ proveniência: sim (40610 linhas)


/tmp/ipykernel_38962/123103405.py:10: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset=nome_base, state=uf, year=ano)
/tmp/ipykernel_38962/123103405.py:20: PySUSWarning: pysus.sim() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sim(...), pysus.dadosgov.sim(...), or pysus.saude.sim(...). Behavior is unchanged.
  caminhos = funcao(state=uf, year=ano)
DORS2024.parquet: 0.00B [00:00, ?B/s]

DO24OPEN.parquet: 0.00B [00:00, ?B/s]
DORS2024.parquet: 0.00B [00:00, ?B/s]


  ✓ proveniência: sim (101480 linhas)
Baixando sim - MS - 2024...


/tmp/ipykernel_38962/123103405.py:10: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset=nome_base, state=uf, year=ano)
/tmp/ipykernel_38962/123103405.py:20: PySUSWarning: pysus.sim() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sim(...), pysus.dadosgov.sim(...), or pysus.saude.sim(...). Behavior is unchanged.
  caminhos = funcao(state=uf, year=ano)
DO24OPEN.parquet: 0.00B [00:00, ?B/s]

DO24OPEN.parquet: 0.00B [00:00, ?B/s]
DOMS2024.parquet: 0.00B [00:00, ?B/s]
/tmp/ipykernel_38962/123103405.py:10: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_

...

  ✓ proveniência: sim (22803 linhas)
Baixando sim - GO - 2024...


/tmp/ipykernel_38962/123103405.py:20: PySUSWarning: pysus.sim() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sim(...), pysus.dadosgov.sim(...), or pysus.saude.sim(...). Behavior is unchanged.
  caminhos = funcao(state=uf, year=ano)
DOGO2024.parquet: 0.00B [00:00, ?B/s]

DO24OPEN.parquet: 0.00B [00:00, ?B/s]
DOGO2024.parquet: 0.00B [00:00, ?B/s]


  ✓ proveniência: sim (47830 linhas)
Baixando sim - DF - 2024...


/tmp/ipykernel_38962/123103405.py:10: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset=nome_base, state=uf, year=ano)
/tmp/ipykernel_38962/123103405.py:20: PySUSWarning: pysus.sim() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sim(...), pysus.dadosgov.sim(...), or pysus.saude.sim(...). Behavior is unchanged.
  caminhos = funcao(state=uf, year=ano)
DO24OPEN.parquet: 0.00B [00:00, ?B/s]

DO24OPEN.parquet: 0.00B [00:00, ?B/s]
DODF2024.parquet: 0.00B [00:00, ?B/s]


  ✓ proveniência: sim (15878 linhas)
40,610 óbitos registrados em AM, 2023-2024
87 colunas disponíveis
101,480 óbitos registrados em RS, 2024
87 colunas disponíveis
106,352 óbitos registrados em MS, MT, GO, DF, 2024
88 colunas disponíveis


In [ ]:
'''
!git add -A
!git commit -m "Fonte 1: SIM (Sistema de Informações de Mortalidade)"
!git push

[main bec84d9] Fonte 1: SIM (Sistema de Informações de Mortalidade)
 8 files changed, 6 insertions(+)
 create mode 100644 dados_brutos/DOAM2023.parquet
 create mode 100644 dados_brutos/DOAM2024.parquet
 create mode 100644 dados_brutos/DODF2024.parquet
 create mode 100644 dados_brutos/DOGO2024.parquet
 create mode 100644 dados_brutos/DOMS2024.parquet
 create mode 100644 dados_brutos/DOMT2024.parquet
 create mode 100644 dados_brutos/DORS2024.parquet
remote: Permission to victorbaraunaAcad/crise-saude-ufam.git denied to gabrielsanttos.
fatal: unable to access 'https://github.com/victorbaraunaAcad/crise-saude-ufam.git/': The requested URL returned error: 403


In [ ]:
#Segunda coleta: SIH(Sistema de Informações Hospitalares)

#Primeiro verificamos os meses disponíveis para download para cada localidade e ano

def meses_com_internacoes(uf,ano):
  catalogo = list_files(dataset="sih", state=uf, year=ano)
  # O nome do arquivo é do tipo RDPR2408.parquet: grupo(2) + UF(2) + ano(2) + mês(2)
  meses_rd = sorted(
    int(nome.split( "\\" ) [-1][6:8])
    for nome in catalogo["name"]
    if nome.split("\\")[-1].startswith("RD")
)

  print(f"Meses com internações (grupo RD) em {uf}/{ano}: {meses_rd}")

  if not meses_rd:
      print("⚠ Nenhum mês publicado para este ano. Tente o ano anterior.")
  else:
      MES = meses_rd[-1]     # usamos o mês mais recente disponível



meses_com_internacoes("AM", 2023)
meses_com_internacoes("AM", 2024)
meses_com_internacoes("RS", 2024)
meses_com_internacoes("MS", 2024)
meses_com_internacoes("MT", 2024)
meses_com_internacoes("GO", 2024)
meses_com_internacoes("DF", 2024)


def baixa_meses_com_internacoes_uma_uf(uf, ano):
    """Baixa todos os meses de internação (RD) disponíveis para uma UF/ano e consolida em um único arquivo."""
    catalogo = list_files(dataset="sih", state=uf, year=ano)

    meses_rd = sorted(
        int(nome.split("\\")[-1][6:8])
        for nome in catalogo["name"]
        if nome.split("\\")[-1].startswith("RD")
    )
    if not meses_rd:
        raise ValueError(f"Nenhum mês com arquivo RD encontrado para {uf} em {ano}.")

    urls = [
        f"ftp://ftp.datasus.gov.br/{n}"
        for n in catalogo.loc[catalogo["name"].str.split("\\").str[-1].str.startswith("RD"), "name"]
    ]

    caminhos = sih(state=uf, year=ano, month=meses_rd)
    arquivos_rd = [c for c in caminhos if os.path.basename(str(c)).upper().startswith("RD")]
    if not arquivos_rd:
        raise ValueError(f"Nenhum arquivo RD baixado para {uf} em {ano}.")

    internacoes = pd.concat(
        [pd.read_parquet(str(c).replace("\\", "/")) for c in arquivos_rd],
        ignore_index=True
    )

    print(f"{len(internacoes):,} internações em {uf}, meses {meses_rd} de {ano}")
    print(f"{len(internacoes.columns)} colunas")

    for origem in arquivos_rd:
        destino = RAIZ_BRUTOS / pathlib.Path(str(origem)).name
        shutil.copy(origem, destino)

    arquivo_bruto = RAIZ_BRUTOS / f"sih_{uf.lower()}_{ano}.parquet"
    internacoes.to_parquet(arquivo_bruto, index=False)

    registrar(
        fonte="SIH",
        url="; ".join(urls),
        metodo="sih",
        parametros={"state": uf, "year": ano, "month": meses_rd},
        n_linhas=len(internacoes),
        arquivo_bruto=str(arquivo_bruto),
    )

    return internacoes


def baixa_meses_com_internacoes(ufs, ano):
    """Aceita uma UF (string) ou uma lista de UFs, e devolve tudo já concatenado."""
    if isinstance(ufs, str):
        ufs = [ufs]

    dfs = []
    for uf in ufs:
        print(f"Baixando internações - {uf} - {ano}...")
        df_uf = baixa_meses_com_internacoes_uma_uf(uf, ano)
        df_uf["UF"] = uf
        dfs.append(df_uf)

    return pd.concat(dfs, ignore_index=True)



/tmp/ipykernel_38962/31135269.py:6: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)


Meses com internações (grupo RD) em AM/2023: [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em AM/2024: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em RS/2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em MS/2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em MT/2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em GO/2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses com internações (grupo RD) em DF/2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [ ]:
#Coletar os dados das internações
from pysus import sih
# ATENÇÃO: sih() devolve QUATRO arquivos — RD (as internações), SP (serviços
# profissionais), RJ e ER (rejeitadas). O SP tem uma linha por PROCEDIMENTO, e
# num mês do Paraná são 1,09 milhão de linhas contra 81 mil internações. Ler
# tudo junto inflava a contagem em 13 vezes e derrubava a letalidade de 4,5%
# para 0,3%, sem erro nenhum na tela.

#Primeira coleta: UF = Amazonas, Período: [2023,2024]
internacoes_am = baixa_meses_com_internacoes_uma_uf("AM",[2023,2024])

#Segunda coleta: UF = Rio Grande do Sul, Período: 2024
internacoes_rs = baixa_meses_com_internacoes_uma_uf("RS",2024)

#Terceira coleta: UFs= Mato Grosso do Sul, Mato Grosso, Goiás, Distrito Federal, Período = 2024
UF = ["MS","MT", "GO", "DF"]
ANO = 2024

internacoes_centro_oeste = baixa_meses_com_internacoes(UF, ANO)



/tmp/ipykernel_38962/31135269.py:34: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)
/tmp/ipykernel_38962/31135269.py:49: PySUSWarning: pysus.sih() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sih(...), pysus.dadosgov.sih(...), or pysus.saude.sih(...). Behavior is unchanged.
  caminhos = sih(state=uf, year=ano, month=meses_rd)
RJAM2310.parquet: 0.00B [00:00, ?B/s]

RJAM2406.parquet: 0.00B [00:00, ?B/s]


RJAM2410.parquet: 0.00B [00:00, ?B/s]



RJAM2310.parquet: 0.00B [00:00, ?B/s]
RJAM2406.parquet: 0.00B [00:00, ?B/s]

RJAM2401.parquet: 0.00B [00:00, ?B/s]

ERAM2307.parquet: 0.00B [00:00, ?B/s]



RJAM2401.parquet: 0.00B [00:00, ?B/s]
RJAM2408.parquet: 0.00B [00:00, ?B/s]

RJAM2402.parquet: 0.00B [00:00, ?B/s

403,313 internações em AM, meses [1, 1, 2, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12] de [2023, 2024]
113 colunas
  ✓ proveniência: SIH (403313 linhas)


/tmp/ipykernel_38962/31135269.py:34: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)
/tmp/ipykernel_38962/31135269.py:49: PySUSWarning: pysus.sih() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sih(...), pysus.dadosgov.sih(...), or pysus.saude.sih(...). Behavior is unchanged.
  caminhos = sih(state=uf, year=ano, month=meses_rd)
RJRS2402.parquet: 0.00B [00:00, ?B/s]

ERRS2402.parquet: 0.00B [00:00, ?B/s]


ERRS2409.parquet: 0.00B [00:00, ?B/s]



ERRS2402.parquet: 0.00B [00:00, ?B/s]
RJRS2402.parquet: 0.00B [00:00, ?B/s]

RJRS2409.parquet: 0.00B [00:00, ?B/s]

RJRS2409.parquet: 0.00B [00:00, ?B/s]

RDRS2410.parquet: 0.00B [00:00, ?B/s]
RDRS2409.parquet: 0.00B [00:00, ?B/s]


SPRS2412.parquet: 0.00B [00:00, ?B/s]

831,884 internações em RS, meses [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] de 2024
113 colunas
  ✓ proveniência: SIH (831884 linhas)
Baixando internações - MS - 2024...


/tmp/ipykernel_38962/31135269.py:34: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)
/tmp/ipykernel_38962/31135269.py:49: PySUSWarning: pysus.sih() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sih(...), pysus.dadosgov.sih(...), or pysus.saude.sih(...). Behavior is unchanged.
  caminhos = sih(state=uf, year=ano, month=meses_rd)
ERMS2404.parquet: 0.00B [00:00, ?B/s]

RDMS2407.parquet: 0.00B [00:00, ?B/s]


ERMS2404.parquet: 0.00B [00:00, ?B/s]

RDMS2407.parquet: 0.00B [00:00, ?B/s]
SPMS2408.parquet: 0.00B [00:00, ?B/s]


ERMS2411.parquet: 0.00B [00:00, ?B/s]


RDMS2406.parquet: 0.00B [00:00, ?B/s]

ERMS2411.parquet: 0.00B [00:00, ?B/s]
RJMS2402.parquet: 0.00B [00:00, ?B/s]


SPMS2412.parquet: 0.00B [00:00, ?B/s]

...

248,618 internações em MT, meses [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] de 2024
113 colunas
  ✓ proveniência: SIH (248618 linhas)
Baixando internações - GO - 2024...


/tmp/ipykernel_38962/31135269.py:34: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)
/tmp/ipykernel_38962/31135269.py:49: PySUSWarning: pysus.sih() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sih(...), pysus.dadosgov.sih(...), or pysus.saude.sih(...). Behavior is unchanged.
  caminhos = sih(state=uf, year=ano, month=meses_rd)
SPGO2403.parquet: 0.00B [00:00, ?B/s]

RJGO2412.parquet: 0.00B [00:00, ?B/s]


RJGO2412.parquet: 0.00B [00:00, ?B/s]


SPGO2412.parquet: 0.00B [00:00, ?B/s]
SPGO2403.parquet: 0.00B [00:00, ?B/s]

RJGO2402.parquet: 0.00B [00:00, ?B/s]


RJGO2402.parquet: 0.00B [00:00, ?B/s]

SPGO2401.parquet: 0.00B [00:00, ?B/s]


RDGO2411.parquet: 0.00B [00:00, ?B/s]



ERGO2401.parquet: 0.00B [00:00, ?B

452,002 internações em GO, meses [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] de 2024
113 colunas
  ✓ proveniência: SIH (452002 linhas)
Baixando internações - DF - 2024...


/tmp/ipykernel_38962/31135269.py:34: PySUSWarning: pysus.list_files() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.list_files(...), pysus.dadosgov.list_files(...), or pysus.saude.list_files(...). Behavior is unchanged.
  catalogo = list_files(dataset="sih", state=uf, year=ano)
/tmp/ipykernel_38962/31135269.py:49: PySUSWarning: pysus.sih() is deprecated and will be removed. Use the origin-namespaced API instead, e.g. pysus.ftp.sih(...), pysus.dadosgov.sih(...), or pysus.saude.sih(...). Behavior is unchanged.
  caminhos = sih(state=uf, year=ano, month=meses_rd)
ERDF2405.parquet: 0.00B [00:00, ?B/s]

ERDF2402.parquet: 0.00B [00:00, ?B/s]


ERDF2402.parquet: 0.00B [00:00, ?B/s]


ERDF2405.parquet: 0.00B [00:00, ?B/s]
RJDF2401.parquet: 0.00B [00:00, ?B/s]

RJDF2411.parquet: 0.00B [00:00, ?B/s]


ERDF2404.parquet: 0.00B [00:00, ?B/s]


RDDF2412.parquet: 0.00B [00:00, ?B/s]
RJDF2411.parquet: 0.00B [00:00, ?B/s]

SPDF2406.parquet: 0.00B [00:00, ?B/s]

248,088 internações em DF, meses [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] de 2024
113 colunas
  ✓ proveniência: SIH (248088 linhas)


In [ ]:
'''
!git add -A
!git commit -m "Fonte 2: SIH(Sistema de Informações Hospitalares)"
!git push

In [ ]:
#Terceiro banco de coleta: SINAN(Sistema de Informação de Agravos de Notificação), mais específico para o caso do RS
#Definição das funções que serão utilizadas
#Os dados do SINAN ajudam a ter uma visão sobre casos de doenãs de agravo de notificação em RS

def mil(numero):
  "Número no padrão brasileiro: 1234567 vira 1.234.567."
  return f"{numero:,.0f}".replace(",", ".")


def _arquivo_legivel(caminho):
    "Confere se TODOS os blocos do parquet abrem — count(*) não basta."
    try:
        leitor = pq.ParquetFile(caminho)
        for grupo in range(leitor.metadata.num_row_groups):
            leitor.read_row_group(grupo)
        return True
    except Exception:
        return False


def _datas_convertem(caminho, coluna="DT_SIN_PRI"):
    """Confere se a coluna de data está em formato ISO e não compacto."""
    try:
      caminho_sql = str(caminho).replace("\\", "/")
      total, iso_ok, compacto_ok = duckdb.sql(f"""
                  SELECT
                  count(*),
                  count(TRY_CAST({coluna} AS DATE)),
                  count(TRY_STRPTIME({coluna}, '%Y%m%d'))
              FROM (SELECT {coluna} FROM read_parquet('{caminho_sql}') LIMIT 5000)
          """).fetchone()
    except Exception:
          return True
    if total == 0:
          return True
    return max(iso_ok, compacto_ok) >= total * 0.9


def arquivo_nacional_confiavel(sigla, ano):
    """Baixa (por fluxo único) e confere o arquivo nacional do agravo no SINAN.
    Contorna o defeito medido (31/08–01/09/2026) de corrida entre duas
    origens duplicadas do catálogo. Retorna (caminho_local, caminho_remoto)."""
    import pysus

    catalogo = pysus.list_files(dataset="sinan", year=ano)
    catalogo = catalogo.assign(arquivo=[
        os.path.basename(str(n).replace("\\", "/")) for n in catalogo["name"]])
    candidatos = catalogo[catalogo["arquivo"].str.upper()
                          .str.startswith(sigla.upper())]
    if candidatos.empty:
        raise FileNotFoundError(f"sem arquivo nacional de {sigla} para {ano}")

    preferida = candidatos[candidatos["path"].astype(str).str.contains("dadosgov")]
    escolhido = (preferida if len(preferida) else candidatos).iloc[0]

    destino = (Path(pysus.CACHEPATH) / "downloads" / "ducklake" / "sinan"
               / escolhido["arquivo"])
    destino.parent.mkdir(parents=True, exist_ok=True)

    if not (destino.exists() and _arquivo_legivel(destino)
            and _datas_convertem(destino)):
        motivo = ("a cópia local estava corrompida ou com datas compactas"
                  if destino.exists() else "primeira vez nesta máquina")
        print(f"  Baixando {escolhido['arquivo']} por fluxo único ({motivo})…")
        from pysus.api import types
        origem = (f"https://{types.S3_ENDPOINT}/{types.S3_BUCKET}/"
                  + str(escolhido["path"]).replace("\\", "/"))
        with httpx.Client(timeout=httpx.Timeout(20, read=180)) as cliente, \
             cliente.stream("GET", origem) as resposta, \
             open(destino, "wb") as saida:
            resposta.raise_for_status()
            for pedaco in resposta.iter_bytes(chunk_size=1024 * 1024):
                saida.write(pedaco)
        if not _datas_convertem(destino):
            raise RuntimeError(
                f"{escolhido['arquivo']} veio com datas compactas mesmo da "
                "origem preferida — as duas origens divergiram de novo.")
        if not _arquivo_legivel(destino):
            raise RuntimeError(
                f"{escolhido['arquivo']} veio ilegível mesmo por fluxo único — "
                "a origem está servindo o arquivo corrompido. Tente mais tarde.")

    caminho_remoto = f"https://.../{str(escolhido['path']).replace(chr(92)+chr(92), '/')}"
    return str(destino).replace("\\", "/"), str(escolhido["path"]).replace("\\", "/")


def baixar_sinan_doencas(doencas, ano):
    """Baixa, confere e consolida o SINAN para um conjunto de doenças.

    doencas: dict {sigla: nome legível}, ex.: {"LEPT": "Leptospirose", "DENG": "Dengue"}
    Salva o bruto conferido + o consolidado padronizado em RAIZ_BRUTOS,
    e registra a proveniência de cada doença.
    """
    from pysus.api import types

    resultados = {}
    for sigla, nome in doencas.items():
        print(f"Baixando SINAN - {nome} ({sigla}) - {ano}...")
        caminho_local, caminho_remoto = arquivo_nacional_confiavel(sigla, ano)
        df = pd.read_parquet(caminho_local)
        print(f"  {len(df):,} registros, {len(df.columns)} colunas")

        # copia o arquivo original já conferido para dados_brutos/
        destino_bruto = RAIZ_BRUTOS / os.path.basename(caminho_local)
        shutil.copy(caminho_local, destino_bruto)

        # salva também a versão consolidada com nome padronizado
        arquivo_bruto = RAIZ_BRUTOS / f"sinan_{sigla.lower()}_{ano}.parquet"
        df.to_parquet(arquivo_bruto, index=False)

        registrar(
            fonte=f"SINAN - {nome}",
            url=f"https://{types.S3_ENDPOINT}/{types.S3_BUCKET}/{caminho_remoto}",
            metodo="arquivo_nacional_confiavel (fluxo único, contorna corrida do catálogo)",
            parametros={"disease": sigla, "year": ano},
            n_linhas=len(df),
            arquivo_bruto=str(arquivo_bruto),
        )

        resultados[sigla] = df

    return resultados



In [ ]:
##A função sinan() do PySUS só aceita disease e year como parâmetros — não tem state nenhum, diferente de sih()/sim(), que aceitam state.
#Isso não é limitação do PySUS, é reflexo de como o DATASUS disponibiliza o dado: o SINAN é publicado como um arquivo único por agravo, cobrindo o Brasil inteiro.


DOENCAS = {"LEPT": "Leptospirose", "DENG": "Dengue", "ZIKAB": "Zika", "CHIKB": "Chikungunya"}
dados_sinan = baixar_sinan_doencas(DOENCAS, 2024)

df_lept = dados_sinan["LEPT"]

In [ ]:
'''
!git add -A
!git commit -m "Fonte 3: SINAN (Sistema de Informação de Agravos de Notificação)"
!git push

### Funções de Verificação e Inspeção de Dados

Criaremos funções para verificar rapidamente a integridade das tabelas coletadas.

In [ ]:
def analisar_tabela(df, nome_tabela):
    """Exibe informações básicas, dimensões e primeiras linhas de um DataFrame."""
    print(f"=== Análise da Tabela: {nome_tabela} ===")
    print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas\n")

    # Tipos de dados e valores não-nulos
    print("--- Resumo de Tipos e Completude (Amostra de colunas) ---")
    info_df = pd.DataFrame({
        'Tipo': df.dtypes,
        'Preenchidos (%)': (df.notna().sum() / len(df) * 100).round(2),
        'Valores Únicos': df.nunique()
    })
    display(info_df.head(15))
    print("\n--- Primeiras 5 linhas ---")
    display(df.head(5))
    print("=" * 50 + "\n")

def verificar_cobertura_temporal(df, coluna_data, formato="%Y%m%d"):
    """Analisa o intervalo temporal coberto pelos dados na tabela."""
    try:
        datas = pd.to_datetime(df[coluna_data], format=formato, errors='coerce')
        print(f"Intervalo Temporal ({coluna_data}):")
        print(f"  Mínimo: {datas.min()}")
        print(f"  Máximo: {datas.max()}")
        print(f"  Registros nulos na data: {datas.isna().sum():,}")
    except Exception as e:
        print(f"Erro ao analisar datas na coluna {coluna_data}: {e}")

#### Exemplo de uso das funções nas tabelas coletadas do SINAN

In [ ]:
# Exemplo de verificação na tabela de Leptospirose
analisar_tabela(df_lept, "SINAN - Leptospirose 2024")
verificar_cobertura_temporal(df_lept, "DT_NOTIFIC")

#### Verificação das Tabelas SIM (Sistema de Informações sobre Mortalidade)

In [ ]:
# Verificação na tabela de Óbitos do Amazonas
analisar_tabela(obitos_am, "SIM - Óbitos Amazonas 2023-2024")
verificar_cobertura_temporal(obitos_am, "DT_OBITO")

In [ ]:
# Verificação na tabela de Óbitos do Rio Grande do Sul
analisar_tabela(obitos_rs, "SIM - Óbitos Rio Grande do Sul 2024")
verificar_cobertura_temporal(obitos_rs, "DT_OBITO")

In [ ]:
# Verificação na tabela de Óbitos do Centro-Oeste
analisar_tabela(obitos_centro_oeste, "SIM - Óbitos Centro-Oeste 2024")
verificar_cobertura_temporal(obitos_centro_oeste, "DT_OBITO")

#### Verificação das Tabelas SIH (Sistema de Informações Hospitalares)

In [ ]:
# Verificação na tabela de Internações do Amazonas
analisar_tabela(internacoes_am, "SIH - Internações Amazonas")
verificar_cobertura_temporal(internacoes_am, "DT_INTERNA")

In [ ]:
# Verificação na tabela de Internações do Rio Grande do Sul
analisar_tabela(internacoes_rs, "SIH - Internações Rio Grande do Sul 2024")
verificar_cobertura_temporal(internacoes_rs, "DT_INTERNA")

In [ ]:
# Verificação na tabela de Internações do Centro-Oeste
analisar_tabela(internacoes_centro_oeste, "SIH - Internações Centro-Oeste 2024")
verificar_cobertura_temporal(internacoes_centro_oeste, "DT_INTERNA")

#### Verificação das demais Tabelas SINAN (Sistema de Informação de Agravos de Notificação)

In [ ]:
# Verificação na tabela de Dengue
analisar_tabela(dados_sinan["DENG"], "SINAN - Dengue 2024")
verificar_cobertura_temporal(dados_sinan["DENG"], "DT_NOTIFIC")

In [ ]:
# Verificação na tabela de Zika
analisar_tabela(dados_sinan["ZIKAB"], "SINAN - Zika 2024")
verificar_cobertura_temporal(dados_sinan["ZIKAB"], "DT_NOTIFIC")

In [ ]:
# Verificação na tabela de Chikungunya
analisar_tabela(dados_sinan["CHIKB"], "SINAN - Chikungunya 2024")
verificar_cobertura_temporal(dados_sinan["CHIKB"], "DT_NOTIFIC")